## ============================================================================

## E-COMMERCE FUNNEL ANALYTICS PROJECT
## NOTEBOOK: 00_DATA_QUALITY

### PURPOSE:
### Assess the quality and consistency of the source e-commerce event data
### before preparing the dataset and calculating funnel metrics.

## ============================================================================

## Analysis objectives

1. Load and preview the source event dataset.
2. Review the dataset structure and column data types.
3. Check missing and blank values.
4. Validate event type values.
5. Check for duplicated rows.
6. Validate the event timestamp range.

In [11]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Preview source data

The source file contains event-level records. Each row represents an
interaction made by a user during an e-commerce session.

In [10]:
events = pd.read_csv("../data/events.csv")
events.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,NaN,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,NaN,NaN,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,NaN,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ


## Count total source records

I will first confirm the total number of event records and columns available
for the analysis.

In [15]:
rows, columns = events.shape

print(f"Total source records: {rows}")
print(f"Total columns: {columns}")

Total source records: 885129
Total columns: 9


## Review source columns

The available fields describe the event timestamp, event type, product,
category, brand, price, user, and session.

In [17]:
print(list(events.columns))

['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


## Review data types and non-null values

`info()` helps identify the current data type of each field and the number
of non-null values. 
The timestamp will be converted to a datetime field
during the data preparation stage.

In [5]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 885129 entries, 0 to 885128
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     885129 non-null  str    
 1   event_type     885129 non-null  str    
 2   product_id     885129 non-null  int64  
 3   category_id    885129 non-null  int64  
 4   category_code  648910 non-null  str    
 5   brand          672765 non-null  str    
 6   price          885129 non-null  float64
 7   user_id        885129 non-null  int64  
 8   user_session   884964 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 60.8 MB


## NULL and blank value validation

I will quantify missing values before deciding how each field should be
treated. Missing category or brand information may be retained as
`Unknown`, while missing session identifiers require special attention for
session-level funnel metrics.

In [33]:
missing_values = events.isna().sum().to_frame("missing_values")
missing_values["missing_percentage"] = (
    missing_values["missing_values"] / len(events) * 100
)

missing_values.sort_values("missing_values", ascending=False)

,missing_values,missing_percentage
category_code,236219,26.69
brand,212364,23.99
user_session,165,0.02
event_time,0,0.00
event_type,0,0.00
category_id,0,0.00
product_id,0,0.00
price,0,0.00
user_id,0,0.00


## Validate session-user consistency

A reliable session identifier should normally be associated with one user. I will check whether any `user_session` value is linked to multiple `user_id` values. This helps identify possible session identifier reuse or collisions before calculating session-level funnel metrics.

In [29]:
session_user_counts = (
    events.groupby("user_session")["user_id"]
    .nunique()
)

session_user_conflicts = (
    session_user_counts > 1
)

print(
    f"Session identifiers linked to multiple users: {session_user_conflicts.sum():,}"
)

session_user_counts[session_user_conflicts].head()

Session identifiers linked to multiple users: 214


user_session
03a42fdd-1f46-4abc-a473-6f3e23f5c566    2
0A7NmM6Igg                              2
0AmYJDGcAZ                              2
0IRUHTsbql                              2
0fI8UfZjSU                              2
Name: user_id, dtype: int64

## Check event type values

The funnel is expected to contain three event types: product views, cart
actions, and purchases. I will validate the values and their proportions.

In [7]:
event_counts = events["event_type"].value_counts(dropna=False).to_frame("event_count")
event_counts["event_percentage"] = event_counts["event_count"] / len(events) * 100
event_counts

,event_count,event_percentage
event_type,,
view,793748,89.68
cart,54035,6.10
purchase,37346,4.22


## Check fully duplicated rows

I will count fully duplicated records. I will not remove them in this
notebook because the cleaning decision belongs in the data preparation
stage and should be documented there.

In [37]:
fully_duplicated_rows = events.duplicated().sum()
print(f"Fully duplicated rows: {fully_duplicated_rows:}")

events[events.duplicated(keep=False)].head(6)

Fully duplicated rows: 655


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
511,2020-09-24 13:51:07 UTC,view,387956,2144415922427789416,computers.components.videocards,asus,104.21,1515915625519429853,PZu2caZ5EN
512,2020-09-24 13:51:07 UTC,view,387956,2144415922427789416,computers.components.videocards,asus,104.21,1515915625519429853,PZu2caZ5EN
974,2020-09-24 15:48:55 UTC,view,874667,2144415922738167921,computers.components.cdrw,asus,23.48,1515915625519457150,8wvs0vbHtv
975,2020-09-24 15:48:55 UTC,view,874667,2144415922738167921,computers.components.cdrw,asus,23.48,1515915625519457150,8wvs0vbHtv
4827,2020-09-25 13:15:09 UTC,view,453469,2144415924222951574,auto.accessories.parktronic,NaN,69.84,1515915625519725870,9ofICyh8Eo
4828,2020-09-25 13:15:09 UTC,view,453469,2144415924222951574,auto.accessories.parktronic,NaN,69.84,1515915625519725870,9ofICyh8Eo


## Check dataset date range

The timestamp is currently stored as text. I will parse it as UTC and check
for invalid values and the period covered by the source data.

In [38]:
event_time_parsed = pd.to_datetime(
    events["event_time"],
    utc=True,
    errors="coerce"
)

invalid_timestamp_count = event_time_parsed.isna().sum()

print(f"Invalid timestamps: {invalid_timestamp_count:}")
print(f"Earliest event: {event_time_parsed.min()}")
print(f"Latest event: {event_time_parsed.max()}")

Invalid timestamps: 0
Earliest event: 2020-09-24 11:57:06+00:00
Latest event: 2021-02-28 23:59:09+00:00
